# Bronze → Silver: Purchase Orders

Transforms noisy bronze `purchase_orders` into a cleaned `silver_purchase_orders` Delta table.

**Bronze characteristics** (see `docs/reference/data-quality-and-noise.md`):
- Append-only rows with `bronze_row_id` as the technical primary key
- Duplicate `purchase_order_id` values (~1% duplicate noise)
- Free-text dates, quantities, and statuses
- ~6% row-level defects: fake suppliers, null fields, bad dates/statuses/quantities, whitespace

**Silver goals**:
1. One canonical row per `purchase_order_id` (latest `bronze_row_id` wins)
2. Trim and normalize string fields
3. Parse typed `order_date`, `expected_delivery_date`, and `quantity`
4. Validate `supplier_id` and `material_id` against master data
5. Normalize status to the ERP vocabulary
6. Add data-quality flags for observability

In [ ]:
CATALOG = "jm_databricks_learning_ws.default"
BRONZE_TABLE = f"{CATALOG}.bronze_purchase_orders"
SILVER_TABLE = f"{CATALOG}.silver_purchase_orders"
SILVER_REJECTS_TABLE = f"{CATALOG}.silver_purchase_orders_rejects"
SUPPLIERS_TABLE = f"{CATALOG}.silver_suppliers"
MATERIALS_TABLE = f"{CATALOG}.silver_materials"

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    coalesce,
    col,
    count,
    current_timestamp,
    length,
    lit,
    regexp_extract,
    row_number,
    sum as sum_,
    to_date,
    trim,
    upper,
    when,
)
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

VALID_STATUSES = ["CREATED", "CONFIRMED", "DELIVERED", "CANCELLED"]

## 1. Load bronze and profile noise

In [ ]:
bronze_df = spark.table(BRONZE_TABLE)
bronze_count = bronze_df.count()
distinct_po_count = bronze_df.select("purchase_order_id").distinct().count()

print(f"Bronze rows: {bronze_count:,}")
print(f"Distinct purchase_order_id: {distinct_po_count:,}")
print(f"Duplicate business keys: {bronze_count - distinct_po_count:,}")

display(bronze_df.groupBy("status").count().orderBy(col("count").desc()))

## 2. Deduplicate by `purchase_order_id`

Bronze allows duplicate `purchase_order_id` rows from the ~1% duplicate noise rate.
Keep the row with the highest `bronze_row_id` (latest ingest).

In [ ]:
dedupe_window = Window.partitionBy("purchase_order_id").orderBy(col("bronze_row_id").desc())

deduped_df = (
    bronze_df
    .withColumn("_row_num", row_number().over(dedupe_window))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

print(f"Rows after dedupe: {deduped_df.count():,}")

## 3. Clean string fields

Handles whitespace noise on `supplier_id` and trims all text identifiers.

In [ ]:
cleaned_df = (
    deduped_df
    .withColumn("purchase_order_id", trim(col("purchase_order_id")))
    .withColumn("supplier_id", trim(col("supplier_id")))
    .withColumn("material_id", trim(col("material_id")))
    .withColumn("status_raw", trim(col("status")))
    .withColumn("order_date_raw", trim(col("order_date")))
    .withColumn("expected_delivery_date_raw", trim(col("expected_delivery_date")))
    .withColumn("quantity_raw", trim(col("quantity")))
)

## 4. Normalize status

Maps known noise variants (`DELIVERD`, `confirmed`, trailing spaces) to valid ERP statuses.
Unknown values are kept as `status_raw` uppercased and flagged later.

In [ ]:
normalized_status = (
    when(col("status_raw") == "DELIVERD", lit("DELIVERED"))
    .when(col("status_raw") == "CONFIRMED ", lit("CONFIRMED"))
    .when(col("status_raw") == "confirmed", lit("CONFIRMED"))
    .when((col("status_raw") == "") | col("status_raw").isNull(), lit(None))
    .otherwise(upper(col("status_raw")))
)

status_df = (
    cleaned_df
    .withColumn("status", normalized_status)
    .withColumn("is_valid_status", col("status").isin(VALID_STATUSES))
)

## 5. Parse dates and quantity

Bronze stores dates and quantities as text. Each format is parsed only when the string matches a regex pattern, so invalid values (e.g. `invalid-date`) become `NULL` without failing the job.

Supported date formats from the noise catalog:
- ISO: `2025-06-15`
- ISO datetime: `2025-06-15T00:00:00`
- European: `15/06/2025`
- Text month: `15-Jun-2025`

In [ ]:
def parse_bronze_date(column_name: str):
    raw = col(column_name)
    iso_prefix = regexp_extract(raw, r"^(\d{4}-\d{2}-\d{2})", 1)
    return coalesce(
        when(raw.rlike(r"^\d{4}-\d{2}-\d{2}$"), to_date(raw, "yyyy-MM-dd")),
        when(raw.rlike(r"^\d{4}-\d{2}-\d{2}T"), to_date(raw, "yyyy-MM-dd'T'HH:mm:ss")),
        when(raw.rlike(r"^\d{2}/\d{2}/\d{4}$"), to_date(raw, "dd/MM/yyyy")),
        when(raw.rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"), to_date(raw, "dd-MMM-yyyy")),
        when(length(iso_prefix) > 0, to_date(iso_prefix, "yyyy-MM-dd")),
    )

typed_df = (
    status_df
    .withColumn("order_date", parse_bronze_date("order_date_raw"))
    .withColumn("expected_delivery_date", parse_bronze_date("expected_delivery_date_raw"))
    .withColumn(
        "quantity",
        when(
            col("quantity_raw").rlike(r"^-?\d+(\.\d+)?$"),
            col("quantity_raw").cast("double").cast(IntegerType()),
        ).otherwise(lit(None)),
    )
    .withColumn("is_valid_order_date", col("order_date").isNotNull())
    .withColumn("is_valid_expected_delivery_date", col("expected_delivery_date").isNotNull())
    .withColumn("is_valid_quantity", (col("quantity").isNotNull()) & (col("quantity") > 0))
    .withColumn(
        "is_delivery_before_order",
        when(
            col("order_date").isNotNull() & col("expected_delivery_date").isNotNull(),
            col("expected_delivery_date") < col("order_date"),
        ).otherwise(lit(False)),
    )
)

## 6. Validate foreign keys against master data

- `supplier_id` must exist in `silver_suppliers` (rejects fake `SUP9xx` noise)
- `material_id` must exist in `silver_materials` as `RAW_MATERIAL` (null material noise is kept but flagged)

In [ ]:
suppliers_df = spark.table(SUPPLIERS_TABLE).select("supplier_id").distinct()
raw_materials_df = (
    spark.table(MATERIALS_TABLE)
    .filter(col("material_type") == "RAW_MATERIAL")
    .select("material_id")
    .distinct()
)

validated_df = (
    typed_df
    .join(suppliers_df.withColumn("_valid_supplier", lit(True)), on="supplier_id", how="left")
    .join(raw_materials_df.withColumn("_valid_material", lit(True)), on="material_id", how="left")
    .withColumn("is_valid_supplier", col("_valid_supplier").isNotNull())
    .withColumn(
        "is_valid_material",
        when(col("material_id").isNull(), lit(False)).otherwise(col("_valid_material").isNotNull()),
    )
    .drop("_valid_supplier", "_valid_material")
)

## 7. Assign record quality and build silver output

In [ ]:
scored_df = (
    validated_df
    .withColumn(
        "has_required_fields",
        col("purchase_order_id").isNotNull()
        & (length(col("purchase_order_id")) > 0)
        & col("supplier_id").isNotNull()
        & col("order_date").isNotNull()
        & col("quantity").isNotNull(),
    )
    .withColumn(
        "record_quality",
        when(
            col("has_required_fields")
            & col("is_valid_status")
            & col("is_valid_supplier")
            & col("is_valid_quantity")
            & col("is_valid_expected_delivery_date"),
            lit("VALID"),
        )
        .when(
            col("purchase_order_id").isNull() | (length(col("purchase_order_id")) == 0),
            lit("REJECTED"),
        )
        .otherwise(lit("PARTIAL")),
    )
    .withColumn("silver_loaded_at", current_timestamp())
)

silver_df = scored_df.select(
    "bronze_row_id",
    "purchase_order_id",
    "order_date",
    "supplier_id",
    "material_id",
    "quantity",
    "expected_delivery_date",
    "status",
    "is_valid_status",
    "is_valid_supplier",
    "is_valid_material",
    "is_valid_quantity",
    "is_valid_order_date",
    "is_valid_expected_delivery_date",
    "is_delivery_before_order",
    "record_quality",
    "silver_loaded_at",
)

silver_accepted_df = silver_df.filter(col("record_quality").isin("VALID", "PARTIAL"))
silver_rejects_df = silver_df.filter(col("record_quality") == "REJECTED")

quality_summary = (
    silver_df.groupBy("record_quality")
    .agg(count(lit(1)).alias("row_count"))
    .orderBy("record_quality")
)

display(quality_summary)

## 8. Data quality breakdown

Counts aligned with known bronze noise patterns from `docs/reference/data-quality-and-noise.md`.

In [ ]:
issue_summary = scored_df.agg(
    sum_(when(~col("is_valid_supplier"), 1).otherwise(0)).alias("invalid_supplier_rows"),
    sum_(when(~col("is_valid_material"), 1).otherwise(0)).alias("missing_or_invalid_material_rows"),
    sum_(when(~col("is_valid_quantity"), 1).otherwise(0)).alias("invalid_quantity_rows"),
    sum_(when(~col("is_valid_status"), 1).otherwise(0)).alias("invalid_status_rows"),
    sum_(when(~col("is_valid_order_date"), 1).otherwise(0)).alias("invalid_order_date_rows"),
    sum_(when(col("is_delivery_before_order"), 1).otherwise(0)).alias("delivery_before_order_rows"),
)

display(issue_summary)

## 9. Write silver tables

In [ ]:
(
    silver_accepted_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

(
    silver_rejects_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_REJECTS_TABLE)
)

print(f"Wrote {silver_accepted_df.count():,} rows to {SILVER_TABLE}")
print(f"Wrote {silver_rejects_df.count():,} rows to {SILVER_REJECTS_TABLE}")

## 10. Preview silver output

In [ ]:
display(spark.table(SILVER_TABLE).orderBy(col("purchase_order_id")).limit(20))